# Домашнее задание 7.1 — остывание стержня

**Зелёный уровень: явная схема FTCS и неявная схема Эйлера**

Рассматривается одномерное уравнение теплопроводности

$$
u_t=u_{xx},\qquad 0\le x\le 1,
$$

с условиями Дирихле $u(0,t)=u(1,t)=0$ и начальным профилем

$$
u(x,0)=\sin(\pi x)+0.3\sin(5\pi x).
$$

Точное решение известно:

$$
u(x,t)=e^{-\pi^2t}\sin(\pi x)+0.3e^{-25\pi^2t}\sin(5\pi x).
$$

**Цель:** сравнить FTCS и неявный Эйлер при $r=\Delta t/\Delta x^2=0.4$, на сетках с 25, 50 и 100 интервалами и в моменты $t=0.001,\ 0.01,\ 0.1$.

## 1. Разностные схемы

Пусть $x_j=j\Delta x$, $j=0,\ldots,N$, а $t_n=n\Delta t$. Для внутренних узлов используется центральная разность

$$
u_{xx}(x_j,t_n)\approx
 \frac{u_{j-1}^n-2u_j^n+u_{j+1}^n}{\Delta x^2}.
$$

**Явная схема FTCS:**

$$
u_j^{n+1}=u_j^n+r\left(u_{j-1}^n-2u_j^n+u_{j+1}^n\right).
$$

**Неявный Эйлер:**

$$
-r u_{j-1}^{n+1}+(1+2r)u_j^{n+1}-r u_{j+1}^{n+1}=u_j^n.
$$

Во всех расчётах базовый шаг выбирается из условия $\Delta t=0.4\Delta x^2$. Чтобы получить значения **точно** в заданные моменты времени, последний шаг до момента наблюдения при необходимости укорачивается. Поэтому его фактическое $r$ не превышает 0.4. Для FTCS это сохраняет условие устойчивости.

Максимальная ошибка на сетке:

$$
E_\infty(N,t)=\max_{j=0,\ldots,N}\left|u_j(t)-u(x_j,t)\right|.
$$

Оценка порядка при удвоении числа интервалов:

$$
p_N=\frac{\log(E_N/E_{2N})}{\log 2}.
$$

Поскольку пространственная аппроксимация имеет порядок 2, а временная — порядок 1, при совместном выборе $\Delta t\propto\Delta x^2$ ожидается суммарный порядок около 2.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Параметры эксперимента
R = 0.4
N_VALUES = [25, 50, 100]       # число пространственных интервалов
T_VALUES = [0.001, 0.01, 0.1] # моменты сравнения

plt.rcParams["figure.figsize"] = (8, 4.8)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25


In [ ]:
def initial_condition(x):
    """Начальное распределение температуры."""
    return np.sin(np.pi * x) + 0.3 * np.sin(5 * np.pi * x)


def exact_solution(x, t):
    """Точное решение задачи Коши."""
    return (np.exp(-np.pi**2 * t) * np.sin(np.pi * x)
            + 0.3 * np.exp(-25 * np.pi**2 * t) * np.sin(5 * np.pi * x))


def thomas_algorithm(lower, diagonal, upper, rhs):
    """Решение трёхдиагональной СЛАУ методом прогонки.

    lower и upper имеют длину n-1, diagonal и rhs — длину n.
    Входные массивы не изменяются.
    """
    n = len(diagonal)
    if n == 0:
        return np.array([], dtype=float)
    if len(rhs) != n or len(lower) != n - 1 or len(upper) != n - 1:
        raise ValueError("Несогласованные размеры диагоналей трёхдиагональной матрицы")

    c_star = np.zeros(max(n - 1, 0), dtype=float)
    d_star = np.zeros(n, dtype=float)

    denom = diagonal[0]
    if abs(denom) < 1e-14:
        raise ZeroDivisionError("Нулевой ведущий элемент в методе прогонки")
    if n > 1:
        c_star[0] = upper[0] / denom
    d_star[0] = rhs[0] / denom

    for i in range(1, n):
        denom = diagonal[i] - lower[i - 1] * c_star[i - 1]
        if abs(denom) < 1e-14:
            raise ZeroDivisionError("Нулевой ведущий элемент в методе прогонки")
        if i < n - 1:
            c_star[i] = upper[i] / denom
        d_star[i] = (rhs[i] - lower[i - 1] * d_star[i - 1]) / denom

    solution = np.empty(n, dtype=float)
    solution[-1] = d_star[-1]
    for i in range(n - 2, -1, -1):
        solution[i] = d_star[i] - c_star[i] * solution[i + 1]
    return solution


def ftcs_step(u, r):
    """Один шаг явной схемы FTCS при u(0,t)=u(1,t)=0."""
    if r > 0.5 + 1e-12:
        raise ValueError(f"FTCS неустойчива: r={r:.4g} > 0.5")
    u_new = np.zeros_like(u)
    u_new[1:-1] = u[1:-1] + r * (u[:-2] - 2.0 * u[1:-1] + u[2:])
    return u_new


def backward_euler_step(u, r):
    """Один шаг неявного Эйлера; СЛАУ решается методом прогонки."""
    n_inner = len(u) - 2
    diagonal = np.full(n_inner, 1.0 + 2.0 * r)
    lower = np.full(n_inner - 1, -r)
    upper = np.full(n_inner - 1, -r)
    u_new = np.zeros_like(u)
    u_new[1:-1] = thomas_algorithm(lower, diagonal, upper, u[1:-1])
    return u_new


def solve_until_time(N, target_time, method, r_nominal=R):
    """Решить задачу до target_time, включая укороченный последний шаг."""
    x = np.linspace(0.0, 1.0, N + 1)
    dx = 1.0 / N
    dt_nominal = r_nominal * dx**2
    u = initial_condition(x)
    # Точное соблюдение граничных условий
    u[0] = 0.0
    u[-1] = 0.0

    time = 0.0
    steps = 0
    while time < target_time - 1e-14:
        dt_step = min(dt_nominal, target_time - time)
        r_step = dt_step / dx**2
        if method == "FTCS":
            u = ftcs_step(u, r_step)
        elif method == "Неявный Эйлер":
            u = backward_euler_step(u, r_step)
        else:
            raise ValueError("Неизвестный метод")
        time += dt_step
        steps += 1

    return x, u, steps, dt_nominal


## 2. Расчёт на трёх сетках

Вычислим оба численных решения независимо для каждого момента наблюдения. Это позволяет сохранить одинаковые заданные моменты времени на всех сетках и не сравнивать значения, взятые с разным временным смещением.

In [ ]:
results = {}
error_rows = []
step_rows = []

for N in N_VALUES:
    dx = 1.0 / N
    for t_target in T_VALUES:
        x = np.linspace(0.0, 1.0, N + 1)
        u_exact = exact_solution(x, t_target)
        for method in ["FTCS", "Неявный Эйлер"]:
            x_num, u_num, n_steps, dt_nominal = solve_until_time(N, t_target, method)
            error = np.max(np.abs(u_num - u_exact))
            results[(N, t_target, method)] = u_num
            error_rows.append({
                "t": t_target,
                "N": N,
                "Метод": method,
                "E_inf": error,
            })
            step_rows.append({
                "t": t_target,
                "N": N,
                "Метод": method,
                "Число шагов": n_steps,
                "dt_nominal": dt_nominal,
            })

errors = pd.DataFrame(error_rows)
steps_info = pd.DataFrame(step_rows)

print("Расчёты завершены.")
print(f"Количество комбинаций (сетка, время, метод): {len(results)}")
print("Номинальное r во всех расчётах:", R)


## 3. Численные и точные профили

На каждом графике показаны точное решение, FTCS и неявный Эйлер. Строки соответствуют сеткам, столбцы — моментам времени.

In [ ]:
for N in N_VALUES:
    x = np.linspace(0.0, 1.0, N + 1)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), sharey=True)
    fig.suptitle(f"Сравнение профилей температуры, N = {N}", fontsize=14)

    for ax, t in zip(axes, T_VALUES):
        u_ex = exact_solution(x, t)
        u_ftcs = results[(N, t, "FTCS")]
        u_be = results[(N, t, "Неявный Эйлер")]

        ax.plot(x, u_ex, color="black", linewidth=2.2, label="Точное решение")
        ax.plot(x, u_ftcs, "o--", markersize=2.8, linewidth=1.1, label="FTCS")
        ax.plot(x, u_be, "s-.", markersize=2.4, linewidth=1.1, label="Неявный Эйлер")
        ax.set_title(f"t = {t:g}")
        ax.set_xlabel("x")
        ax.set_xlim(0, 1)

    axes[0].set_ylabel("u(x,t)")
    axes[-1].legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.show()


## 4. Максимальные ошибки и порядок сходимости

Для каждого момента времени вычисляется ошибка $E_\infty$. В столбце $p$ указана оценка порядка между текущей сеткой $N$ и следующей, более мелкой сеткой $2N$. Для самой мелкой сетки такой оценки нет.

In [ ]:
# Таблица ошибок с оценками p_N = log2(E_N / E_2N)
order_rows = []
for t in T_VALUES:
    for method in ["FTCS", "Неявный Эйлер"]:
        method_errors = {
            int(row.N): float(row.E_inf)
            for row in errors[(errors["t"] == t) & (errors["Метод"] == method)].itertuples()
        }
        for N in N_VALUES:
            E = method_errors[N]
            if 2 * N in method_errors:
                p = np.log(method_errors[N] / method_errors[2 * N]) / np.log(2.0)
            else:
                p = np.nan
            order_rows.append({
                "t": t,
                "Метод": method,
                "N": N,
                "E_inf": E,
                "p (N → 2N)": p,
            })

order_table = pd.DataFrame(order_rows)
order_table["E_inf"] = order_table["E_inf"].map(lambda v: f"{v:.6e}")
order_table["p (N → 2N)"] = order_table["p (N → 2N)"].map(
    lambda v: "—" if pd.isna(v) else f"{v:.3f}"
)

print("Максимальная абсолютная ошибка и оценка порядка сходимости:")
display(order_table)


## 5. Число временных шагов

Ниже приведена справочная таблица числа выполненных шагов. Базовый шаг равен $0.4\Delta x^2$; если целевой момент времени не кратен ему, последний шаг короче.

In [ ]:
steps_display = steps_info.copy()
steps_display["dt_nominal"] = steps_display["dt_nominal"].map(lambda v: f"{v:.6e}")
display(steps_display)


## 6. Ответы на вопросы и выводы

**Почему мелкие детали температурного профиля исчезают быстрее крупных?**

Решение разлагается по собственным гармоникам оператора второй производной. Для гармоники $\sin(k\pi x)$ амплитуда убывает как

$$
A_k(t)=A_k(0)e^{-k^2\pi^2t}.
$$

Скорость затухания пропорциональна $k^2$. Поэтому коротковолновые, быстро меняющиеся по координате детали (большие $k$) сглаживаются значительно быстрее длинноволновых. В данной задаче вторая гармоника начального профиля имеет волновое число $5\pi$, поэтому её амплитуда убывает в 25 раз быстрее по экспоненциальному показателю, чем амплитуда основной гармоники $\sin(\pi x)$.

**Вывод по схемам.**

- FTCS при $r=0.4$ удовлетворяет условию устойчивости для одномерного уравнения теплопроводности $r\le 1/2$.
- Неявный Эйлер устойчив при любом положительном шаге времени, но это само по себе не означает нулевую ошибку: точность зависит от $\Delta x$ и $\Delta t$.
- При выбранном совместном сгущении $\Delta t=0.4\Delta x^2$ пространственная ошибка второго порядка и временная ошибка первого порядка по $\Delta t$ обе имеют масштаб $O(\Delta x^2)$. Поэтому ожидается порядок около 2 для обоих методов.
- Небольшие отклонения наблюдаемого порядка от 2 возможны из-за конечного диапазона сеток, выбранного момента времени и укороченного последнего временного шага.

